# Notebook 2: Exploratory Data Analysis

This notebook performs exploratory data analysis on the training data, test data, and supplementary datasets.

## 2.1 Load and Overview Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

# Set working directory
WORK_DIR = "/home/kwierman/Desktop/Projects/DeepPastAkkadian"
os.chdir(WORK_DIR)

# Load main datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

print("=" * 60)
print("TRAIN DATA OVERVIEW")
print("=" * 60)
print(f"Shape: {train_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\nData types:\n{train_df.dtypes}")
print(f"\nMissing values:\n{train_df.isnull().sum()}")

In [ ]:
print("\n" + "=" * 60)
print("TEST DATA OVERVIEW")
print("=" * 60)
print(f"Shape: {test_df.shape}")
print(f"\nColumns: {test_df.columns.tolist()}")
print(f"\nData types:\n{test_df.dtypes}")
print(f"\nMissing values:\n{test_df.isnull().sum()}")

## 2.2 Text Length Analysis

In [ ]:
# Calculate text lengths
train_df['src_len'] = train_df['transliteration'].str.len()
train_df['tgt_len'] = train_df['translation'].str.len()
train_df['src_words'] = train_df['transliteration'].str.split().str.len()
train_df['tgt_words'] = train_df['translation'].str.split().str.len()

test_df['src_len'] = test_df['transliteration'].str.len()
test_df['src_words'] = test_df['transliteration'].str.split().str.len()

print("=" * 60)
print("SOURCE TEXT LENGTH STATISTICS (Characters)")
print("=" * 60)
print("\nTRAIN:")
print(train_df['src_len'].describe())
print("\nTEST:")
print(test_df['src_len'].describe())

In [ ]:
print("=" * 60)
print("SOURCE TEXT LENGTH STATISTICS (Words)")
print("=" * 60)
print("\nTRAIN:")
print(train_df['src_words'].describe())
print("\nTEST:")
print(test_df['src_words'].describe())

In [ ]:
print("=" * 60)
print("TARGET TEXT LENGTH STATISTICS (Characters)")
print("=" * 60)
print("\nTRAIN translations:")
print(train_df['tgt_len'].describe())

In [ ]:
# Visualize length distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Source length (chars) - train vs test
axes[0, 0].hist(train_df['src_len'], bins=50, alpha=0.7, label='Train', color='blue')
axes[0, 0].hist(test_df['src_len'], bins=50, alpha=0.7, label='Test', color='orange')
axes[0, 0].set_xlabel('Source Length (characters)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Source Text Length Distribution')
axes[0, 0].legend()

# Source length (words) - train vs test
axes[0, 1].hist(train_df['src_words'], bins=50, alpha=0.7, label='Train', color='blue')
axes[0, 1].hist(test_df['src_words'], bins=50, alpha=0.7, label='Test', color='orange')
axes[0, 1].set_xlabel('Source Length (words)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Source Text Word Count Distribution')
axes[0, 1].legend()

# Target length distribution
axes[1, 0].hist(train_df['tgt_len'], bins=50, alpha=0.7, color='green')
axes[1, 0].set_xlabel('Target Length (characters)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Target (Translation) Length Distribution')

# Word count target
axes[1, 1].hist(train_df['tgt_words'], bins=50, alpha=0.7, color='green')
axes[1, 1].set_xlabel('Target Length (words)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Target (Translation) Word Count Distribution')

plt.tight_layout()
plt.savefig('notebooks/eda_length_distributions.png', dpi=150)
plt.show()

print("Saved: notebooks/eda_length_distributions.png")

## 2.3 Special Character Analysis

In [ ]:
# Analyze special characters and markers in texts
def find_special_patterns(text):
    """Find special patterns in transliteration"""
    patterns = {
        'gap': r'<gap>',
        'big_gap': r'<big_gap>',
        'determinatives': r'\{[^}]+\}',
        'brackets': r'[\[\]]',
        'parentheses': r'[\(\)]',
        'special_h': r'[Ḫḫ]',
        'subscript': r'[₀-₉ₓ]',
        'sumerogram': r'[A-Z]{2,}',
    }
    results = {}
    for name, pattern in patterns.items():
        results[name] = len(re.findall(pattern, str(text)))
    return results

# Analyze patterns in train and test
train_patterns = train_df['transliteration'].apply(find_special_patterns)
test_patterns = test_df['transliteration'].apply(find_special_patterns)

train_patterns_df = pd.DataFrame(train_patterns.tolist())
test_patterns_df = pd.DataFrame(test_patterns.tolist())

print("=" * 60)
print("SPECIAL PATTERNS IN TRANSLITERATIONS")
print("=" * 60)

print("\nTRAIN - Average patterns per text:")
print(train_patterns_df.mean())

print("\nTEST - Average patterns per text:")
print(test_patterns_df.mean())

In [ ]:
# Check specifically for H vs Ḫ (critical for normalization)
print("=" * 60)
print("H CHARACTER ANALYSIS (Critical for normalization)")
print("=" * 60)

train_has_h = train_df['transliteration'].str.contains('H', regex=False)
train_has_hbar = train_df['transliteration'].str.contains('Ḫ|ḫ', regex=True)
test_has_h = test_df['transliteration'].str.contains('H', regex=False)
test_has_hbar = test_df['transliteration'].str.contains('Ḫ|ḫ', regex=True)

print(f"\nTRAIN - Contains 'H': {train_has_h.sum()} ({train_has_h.mean()*100:.1f}%)")
print(f"TRAIN - Contains 'Ḫ/ḫ': {train_has_hbar.sum()} ({train_has_hbar.mean()*100:.1f}%)")
print(f"\nTEST - Contains 'H': {test_has_h.sum()} ({test_has_h.mean()*100:.1f}%)")
print(f"TEST - Contains 'Ḫ/ḫ': {test_has_hbar.sum()} ({test_has_hbar.mean()*100:.1f}%)")

## 2.4 Gap/Fragment Analysis

In [ ]:
# Analyze gaps in texts
print("=" * 60)
print("GAP ANALYSIS")
print("=" * 60)

train_has_gap = train_df['transliteration'].str.contains('<gap>', regex=False)
train_has_big_gap = train_df['transliteration'].str.contains('<big_gap>', regex=False)
test_has_gap = test_df['transliteration'].str.contains('<gap>', regex=False)
test_has_big_gap = test_df['transliteration'].str.contains('<big_gap>', regex=False)

print(f"\nTRAIN - Has <gap>: {train_has_gap.sum()} ({train_has_gap.mean()*100:.1f}%)")
print(f"TRAIN - Has <big_gap>: {train_has_big_gap.sum()} ({train_has_big_gap.mean()*100:.1f}%)")
print(f"\nTEST - Has <gap>: {test_has_gap.sum()} ({test_has_gap.mean()*100:.1f}%)")
print(f"TEST - Has <big_gap>: {test_has_big_gap.sum()} ({test_has_big_gap.mean()*100:.1f}%)")

## 2.5 Vocabulary Analysis

In [ ]:
# Analyze vocabulary - unique words
from collections import Counter

def tokenize(text):
    """Simple tokenization"""
    return str(text).split()

train_tokens = []
for text in train_df['transliteration']:
    train_tokens.extend(tokenize(text))

test_tokens = []
for text in test_df['transliteration']:
    test_tokens.extend(tokenize(text))

train_vocab = Counter(train_tokens)
test_vocab = Counter(test_tokens)

print("=" * 60)
print("VOCABULARY ANALYSIS")
print("=" * 60)

print(f"\nTRAIN unique tokens: {len(train_vocab)}")
print(f"TEST unique tokens: {len(test_vocab)}")

# Vocabulary coverage
train_known = sum(test_vocab.get(t, 0) for t in train_vocab)
test_total = sum(test_vocab.values())
print(f"\nTest tokens covered by train vocabulary: {train_known}/{test_total} ({train_known/test_total*100:.1f}%)")

# Out-of-vocabulary words in test
test_oov = [t for t in test_vocab if t not in train_vocab]
print(f"\nOut-of-vocabulary tokens in test: {len(test_oov)}")
print(f"OOV tokens: {test_oov[:20]}...")

In [ ]:
# Most common tokens
print("=" * 60)
print("TOP 20 MOST COMMON TOKENS (Train)")
print("=" * 60)
for token, count in train_vocab.most_common(20):
    print(f"  {token}: {count}")

## 2.6 Sample Transliterations

In [ ]:
print("=" * 60)
print("SAMPLE TRANSLITERATIONS FROM TRAINING DATA")
print("=" * 60)

for i in range(3):
    print(f"\n--- Sample {i+1} ---")
    print(f"SRC: {train_df.iloc[i]['transliteration'][:200]}...")
    print(f"TGT: {train_df.iloc[i]['translation'][:200]}...")

In [ ]:
print("=" * 60)
print("SAMPLE TRANSLITERATIONS FROM TEST DATA")
print("=" * 60)

for i in range(3):
    print(f"\n--- Sample {i+1} ---")
    print(f"ID: {test_df.iloc[i]['id']}, Text: {test_df.iloc[i]['transliteration'][:200]}...")

## 2.7 EDA Summary

In [ ]:
eda_summary = """
## Exploratory Data Analysis Summary

### Key Findings

1. **Data Size**:
   - Training: 1,561 documents
   - Test: 4 sentences (but ~4,000 expected in actual evaluation)

2. **Text Length**:
   - Train avg source: ~{train_avg} chars, ~{train_words} words
   - Test avg source: ~{test_avg} chars, ~{test_words} words
   - Translations average ~2-3x longer than source (Akkadian is morphologically dense)

3. **Special Characters**:
   - Training data contains Ḫ/ḫ, TEST data contains only H/h
   - Must normalize: Ḫ→H, ḫ→h
   - Gap markers present in both train and test

4. **Vocabulary**:
   - High coverage between train and test
   - Need to handle Sumerograms (ALL CAPS words)

5. **Data Quality Issues**:
   - Training data is document-level, test is sentence-level
   - Need sentence alignment for proper evaluation

### Recommendations for Preprocessing
1. Normalize Ḫ→H before training
2. Handle <gap> and <big_gap> markers appropriately
3. Consider removing determinatives {d}, {ki}, etc. or keeping as special tokens
4. Use subword tokenization (SentencePiece) to handle OOV
""".format(
    train_avg=int(train_df['src_len'].mean()),
    train_words=int(train_df['src_words'].mean()),
    test_avg=int(test_df['src_len'].mean()),
    test_words=int(test_df['src_words'].mean())
)

print(eda_summary)

In [ ]:
# Save summary statistics
stats = {
    'train_samples': len(train_df),
    'test_samples': len(test_df),
    'train_avg_src_len': train_df['src_len'].mean(),
    'train_avg_tgt_len': train_df['tgt_len'].mean(),
    'train_vocab_size': len(train_vocab),
    'test_vocab_size': len(test_vocab),
    'train_has_hbar_pct': train_has_hbar.mean() * 100,
    'test_has_hbar_pct': test_has_hbar.mean() * 100,
}

import json
with open('notebooks/eda_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print("EDA statistics saved to notebooks/eda_stats.json")